# 第二章：Prompt Templates（提示词模板）

## 学习目标
- 理解为什么需要提示词模板（对比硬编码 vs 模板）
- 掌握 `ChatPromptTemplate` 的创建与使用
- 学会使用 `MessagesPlaceholder` 插入动态消息列表
- 了解 Partial Variables（部分变量预填充）
- 实现 Few-shot Prompting（少样本提示）

## 0. 环境准备

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv("../.env")
import langchain
print(f"langchain 版本: {langchain.__version__}")

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="kimi-k2.5",
    openai_api_base=os.environ["Qwen_API_BASE"],
    openai_api_key=os.environ["Qwen_API_KEY"],
    temperature=0.7,
)
print("模型初始化完成")

## 1. 对比：硬编码 vs 模板

先看「不用模板」的写法有什么问题：

In [2]:
from langchain_core.messages import SystemMessage, HumanMessage

# ❌ 硬编码方式：每换一个领域就得复制粘贴改一遍
messages_v1 = [
    SystemMessage(content="你是一位网络安全领域的专家，回答要简洁专业。"),
    HumanMessage(content="什么是 SQL 注入？"),
]

messages_v2 = [
    SystemMessage(content="你是一位机器学习领域的专家，回答要简洁专业。"),
    HumanMessage(content="什么是过拟合？"),
]

# 两段代码 90% 相同，只有「领域」和「问题」不同 —— 这就是模板要解决的问题
print("硬编码方式需要重复大量相似代码")

硬编码方式需要重复大量相似代码


In [3]:
from langchain_core.prompts import ChatPromptTemplate

# ✅ 模板方式：用 {variable} 占位，一次定义，多次复用
prompt = ChatPromptTemplate.from_messages([
    ("system", "你是一位{domain}领域的专家，回答要简洁专业。"),
    ("human", "{question}"),
])

# 同一个模板，不同参数
r1 = llm.invoke(prompt.invoke({"domain": "网络安全", "question": "什么是 SQL 注入？"}))
r2 = llm.invoke(prompt.invoke({"domain": "机器学习", "question": "什么是过拟合？"}))

print("【网络安全】", r1.content[:80], "...")
print()
print("【机器学习】", r2.content[:80], "...")

【网络安全】 

# SQL 注入（SQL Injection）

## 定义
SQL 注入是一种代码注入攻击，攻击者通过在用户输入中插入恶意 SQL 语句，操纵后台数据库执 ...

【机器学习】 

# 过拟合（Overfitting）

**定义**：模型在训练集上表现优异，但在测试集/新数据上性能显著下降的现象。

## 核心特征

| 训练集 |  ...


### 刚才发生了什么？

`ChatPromptTemplate` 把提示词变成了 **可参数化的模板**。`{domain}` 和 `{question}` 是占位符，调用 `.invoke()` 时传入字典填充它们，生成完整的消息列表。

好处：
- **一次定义，多次复用**：换领域只需改参数，不用改模板
- **关注点分离**：模板结构和业务数据分开管理
- **可组合**：模板可以和模型、解析器串联成链（第三章）

## 2. ChatPromptTemplate 详解

In [4]:
# 查看模板的元信息
print("输入变量:", prompt.input_variables)
print("消息模板数量:", len(prompt.messages))

输入变量: ['domain', 'question']
消息模板数量: 2


In [11]:
# .invoke() 返回 ChatPromptValue，包含生成的消息列表
result = prompt.invoke({"domain": "数据库", "question": "什么是索引？"})

print(type(result))  # ChatPromptValue
print()
for msg in result.messages:
    print(f"[{msg.type}] {msg.content}")

<class 'langchain_core.prompt_values.ChatPromptValue'>

[system] 你是一位数据库领域的专家，回答要简洁专业。
[human] 什么是索引？


### 两种创建方式

| 方式 | 说明 |
|------|------|
| `from_messages([(role, template), ...])` | 从消息元组列表创建，支持多角色（最灵活） |
| `from_template("...")` | 从单个字符串创建，仅生成一条 HumanMessage（快捷方式） |

In [12]:
# from_template：快捷方式，只需要一条 HumanMessage 时用
simple_prompt = ChatPromptTemplate.from_template(
    "将以下内容翻译成{language}：\n\n{text}"
)

print("输入变量:", simple_prompt.input_variables)

result = simple_prompt.invoke({"language": "英语", "text": "大语言模型正在改变软件开发的方式"})
for msg in result.messages:
    print(f"[{msg.type}] {msg.content}")

输入变量: ['language', 'text']
[human] 将以下内容翻译成英语：

大语言模型正在改变软件开发的方式


## 3. 实战：代码审查模板

In [13]:
# 构建一个可复用的「代码审查」模板
code_review_prompt = ChatPromptTemplate.from_messages([
    ("system", """你是一位资深的 {language} 代码审查者。
请对用户提供的代码进行审查，重点关注：
1. 潜在的 Bug
2. 性能问题
3. 安全隐患

输出格式：先总结，再逐项列出问题及改进建议。"""),
    ("human", "请审查以下代码：\n```{language}\n{code}\n```"),
])

messages = code_review_prompt.invoke({
    "language": "Python",
    "code": """def get_user(id):
    query = f"SELECT * FROM users WHERE id = {id}"
    result = db.execute(query)
    return result[0]""",
})

response = llm.invoke(messages)
print(response.content)



### 代码审查总结

提供的代码 `get_user(id)` 函数存在多个严重问题，主要包括**安全隐患**（SQL注入风险）、**潜在的Bug**（错误处理缺失）和**性能问题**（低效的查询）。以下是详细审查结果和改进建议。

### 逐项问题及改进建议

1. **安全隐患：SQL注入风险**
   - **问题描述**：代码直接使用字符串拼接（f-string）将 `id` 参数插入到SQL查询中。如果 `id` 来自用户输入（如HTTP请求），攻击者可以通过构造恶意输入（如 `' OR 1=1 --`）来篡改SQL查询逻辑，导致数据泄露或破坏。
   - **潜在影响**：可能导致数据库被非法访问、敏感数据泄露或数据被恶意修改。
   - **改进建议**：使用参数化查询（prepared statements）来避免SQL注入。修改代码如下：
     ```python
     def get_user(id):
         query = "SELECT * FROM users WHERE id = %s"  # 使用占位符
         result = db.execute(query, (id,))  # 传递参数作为元组
         return result[0] if result else None  # 添加空结果处理
     ```
     如果使用ORM（如SQLAlchemy），应优先使用其查询构建器。

2. **潜在的Bug：缺乏错误处理**
   - **问题描述**：代码直接返回 `result[0]`，但如果查询没有匹配结果（如 `id` 不存在），`result` 可能为空列表，导致 `IndexError` 异常。
   - **潜在影响**：运行时抛出未处理的异常，使程序崩溃，影响用户体验。
   - **改进建议**：添加空结果检查，并考虑返回 `None` 或抛出自定义异常。修改代码如下：
     ```python
     def get_user(id):
         query = "SELECT * FROM users WHERE id = %s"
         result = db.execute(query, (id,))
         if 

模板的价值在这里就很明显了——换一段代码或换一种语言，只需要改参数，审查的系统指令完全复用。

## 4. MessagesPlaceholder（消息占位符）

`{variable}` 只能替换为字符串。如果要插入 **一组动态数量的消息**（比如对话历史），需要用 `MessagesPlaceholder`。

In [21]:
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage

# 带对话历史的模板
chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "你是一位友好的 AI 助手。"),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}"),
])

# 模拟多轮对话：传入历史消息
messages = chat_prompt.invoke({
    "history": [
        # HumanMessage(content="我叫Claude"),
        # AIMessage(content="你好Claude！有什么可以帮你的吗？"),
    ],
    "input": "你还记得我叫什么吗？",
})

for msg in messages.messages:
    print(f"[{msg.type}] {msg.content}")

[system] 你是一位友好的 AI 助手。
[human] 你还记得我叫什么吗？


In [22]:
# 模型能基于历史「记住」名字
response = llm.invoke(messages)
print(response.content)



抱歉，我没有持久的记忆功能，无法记住之前对话中的信息。每次对话对我来说都是全新的开始。

不过，如果你愿意告诉我你的名字，我很乐意在这次对话中称呼你！请问你怎么称呼？


### 刚才发生了什么？

`MessagesPlaceholder(variable_name="history")` 会被替换为传入的消息列表。最终生成的消息序列是：

```
SystemMessage → HumanMessage("我叫小明") → AIMessage("你好小明...") → HumanMessage("你还记得...")
```

模型看到了完整的对话上下文，所以能正确回答。没有历史时传空列表 `[]` 即可。

这是实现多轮对话的基础，第四章 Memory 会在此之上自动管理历史。

## 5. Partial Variables（部分变量预填充）

模板中某些变量在创建时就已确定（如日期、用户角色），可以用 `.partial()` 预先填入。

In [24]:
from datetime import datetime

prompt_with_date = ChatPromptTemplate.from_messages([
    ("system", "今天是 {date}。你是一位新闻分析师。"),
    ("human", "{question}"),
])

# 预填充日期，后续调用只需提供 question
partial_prompt = prompt_with_date.partial(
    date=datetime.now().strftime("%Y年%m月%d日")
)

print("填充前变量:", prompt_with_date.input_variables)   # ['date', 'question']
print("填充后变量:", partial_prompt.input_variables)      # ['question']

填充前变量: ['date', 'question']
填充后变量: ['question']


In [29]:
# 使用时只需提供剩余变量
messages = partial_prompt.invoke({"question": "最近 AI 领域有哪些值得关注的趋势？"})

for msg in messages.messages:
    print(f"[{msg.type}] {msg.content}")

[system] 今天是 2026年03月25日。你是一位新闻分析师。
[human] 最近 AI 领域有哪些值得关注的趋势？


### 延迟计算

也支持传入函数，每次调用时动态计算值：

In [31]:
# 用函数延迟计算：每次 invoke 时执行函数获取最新时间
dynamic_prompt = prompt_with_date.partial(
    date=lambda: datetime.now().strftime("%Y年%m月%d日 %H:%M")
)

result = dynamic_prompt.invoke({"question": "现在几点？"})
for msg in result.messages:
    print(f"[{msg.type}] {msg.content}")

[system] 今天是 2026年03月25日 12:41。你是一位新闻分析师。
[human] 现在几点？


### 常见问题

- **`partial()` 返回新对象**：原模板不变，`partial()` 返回一个新的模板副本。
- **函数参数必须无参**：`lambda` 或函数不能接受参数，它在每次 `.invoke()` 时被调用。

## 6. Few-shot Prompting（少样本提示）

通过在提示词中提供几个示例，引导模型按期望的格式输出。

### 方式一：直接在模板中写入示例消息

In [32]:
# 在模板中直接嵌入 human/ai 示例对话
few_shot_prompt = ChatPromptTemplate.from_messages([
    ("system", "你是一个将用户描述转换为 JSON 格式的助手。"),
    # 示例 1
    ("human", "张三，25岁，软件工程师"),
    ("ai", '{{"name": "张三", "age": 25, "job": "软件工程师"}}'),
    # 示例 2
    ("human", "李四，30岁，产品经理"),
    ("ai", '{{"name": "李四", "age": 30, "job": "产品经理"}}'),
    # 用户实际输入
    ("human", "{input}"),
])

messages = few_shot_prompt.invoke({"input": "王五，28岁，数据分析师"})
response = llm.invoke(messages)
print(response.content)



{"name": "王五", "age": 28, "job": "数据分析师"}


模型看到了两个「输入 → JSON」的示例，自然会按照相同格式输出。

> 注意：模板中的 `{{` 和 `}}` 是转义写法，表示字面量 `{` 和 `}`，避免被当作占位符。

### 方式二：FewShotChatMessagePromptTemplate

当示例较多或需要动态选择时，用专用类更灵活：

In [34]:
from langchain_core.prompts import FewShotChatMessagePromptTemplate

# 示例数据（可以来自数据库或文件）
examples = [
    {"input": "happy", "output": "sad"},
    {"input": "tall", "output": "short"},
    {"input": "fast", "output": "slow"},
]

# 每个示例的展示模板
example_prompt = ChatPromptTemplate.from_messages([
    ("human", "{input}"),
    ("ai", "{output}"),
])

# 构建 Few-shot 模板
few_shot = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples,
)

# 组合到完整模板中
final_prompt = ChatPromptTemplate.from_messages([
    ("system", "你是一个给出反义词的助手。"),
    few_shot,
    ("human", "{input}"),
])

response = llm.invoke(final_prompt.invoke({"input": "bright"}))
print(response.content)



dark


### 对比两种方式

| | 直接写入 | FewShotChatMessagePromptTemplate |
|------|---------|----------------------------------|
| 适合场景 | 示例少且固定 | 示例多或需要动态选择 |
| 灵活性 | 低 | 高（可配合 ExampleSelector 按相似度选择示例） |
| 代码量 | 少 | 多一点 |

## 7. 模板组合（Composition）

In [35]:
# 模板可以用 + 操作符拼接
system_part = ChatPromptTemplate.from_messages([
    ("system", "你是一位{role}。回答要简洁明了。"),
])

history_part = ChatPromptTemplate.from_messages([
    MessagesPlaceholder(variable_name="history"),
])

user_part = ChatPromptTemplate.from_messages([
    ("human", "{input}"),
])

# 三个模块拼装成一个完整模板
combined = system_part + history_part + user_part
print("输入变量:", combined.input_variables)

messages = combined.invoke({
    "role": "Python 导师",
    "history": [],
    "input": "解释一下装饰器",
})

for msg in messages.messages:
    print(f"[{msg.type}] {msg.content}")

输入变量: ['history', 'input', 'role']
[system] 你是一位Python 导师。回答要简洁明了。
[human] 解释一下装饰器


### 为什么要拆分？

把模板拆成模块化组件后，可以按需组合。比如：
- 不需要对话历史的场景：`system_part + user_part`
- 需要历史的场景：`system_part + history_part + user_part`
- 不同系统指令：替换 `system_part` 即可

这在构建复杂应用时尤其有用。

## 8. 实战：多风格翻译模板

In [36]:
# 综合运用：一个模板支持多语言、多风格翻译
translation_prompt = ChatPromptTemplate.from_messages([
    ("system", """你是一位专业翻译。请将用户提供的文本翻译成{target_language}。
翻译风格：{style}
只输出翻译结果，不要解释。"""),
    ("human", "{text}"),
])

text = "大语言模型的出现彻底改变了人机交互的方式"

# 正式翻译
r1 = llm.invoke(translation_prompt.invoke({
    "target_language": "英语", "style": "正式、学术", "text": text,
}))
print("正式:", r1.content)

# 口语化翻译
r2 = llm.invoke(translation_prompt.invoke({
    "target_language": "英语", "style": "口语化、轻松", "text": text,
}))
print("口语:", r2.content)

# 日语翻译
r3 = llm.invoke(translation_prompt.invoke({
    "target_language": "日语", "style": "自然、地道", "text": text,
}))
print("日语:", r3.content)

正式: 

The emergence of Large Language Models has fundamentally transformed the manner of human-computer interaction.
口语: 

The rise of LLMs has totally changed how we interact with machines.
日语: 

大規模言語モデルの出現は、人とコンピュータの相互作用のあり方を完全に変えた


同一个模板，三种完全不同的输出。这就是模板化的核心价值。

## 总结

本章学习了：
- ✅ 对比了硬编码 vs 模板的区别
- ✅ `ChatPromptTemplate` 的 `from_messages()` 和 `from_template()` 两种创建方式
- ✅ `{variable}` 占位符与 `.invoke()` 填充变量
- ✅ `MessagesPlaceholder` 插入动态消息列表（对话历史）
- ✅ `.partial()` 预填充已知变量（支持函数延迟计算）
- ✅ Few-shot Prompting 的两种实现方式
- ✅ 模板组合（`+` 操作符）

### 速查表

| 概念 | 关键类/方法 | 用途 |
|------|------------|------|
| 聊天模板 | `ChatPromptTemplate` | 生成结构化消息列表 |
| 消息占位符 | `MessagesPlaceholder` | 插入动态数量的历史消息 |
| 部分填充 | `.partial()` | 预先绑定已知变量 |
| 少样本模板 | `FewShotChatMessagePromptTemplate` | 管理和注入示例 |
| 模板组合 | `prompt_a + prompt_b` | 模块化拼装 |

## 下一章

**第三章：LCEL 链式表达式** —— 学习如何用 `|` 管道操作符将模板、模型、解析器串联成完整的工作流。目前我们还在手动调用 `prompt.invoke()` 再传给 `llm.invoke()`，下一章会把这两步合成一行。